# AML Alert Prioritization — optimized solution
DnkCode · Team `2ABB3C78`

This notebook reproduces the frozen optimized model, its audit comparison, and the prediction CSV. The previous solution remains in `final_solution.ipynb`; the baseline artifacts are archived under `experiments/optimization_20260927/baseline/`.

The optimization search used 10,500 development alerts only. Model selection used two repetitions of stratified 3-fold CV. A separate 3,500-alert audit split was excluded from that search. **Development CV is selection-biased.** The original baseline feature selection had already used all training labels; its audit score is therefore a conservative comparison for the challenger, not a pristine held-out baseline estimate. ROC-AUC is a ranking metric, not classification accuracy.

In [ ]:
import json
import sys
from pathlib import Path
root = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'src' / 'optimize.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from src import config
from src.optimize import RUN, load, split_indices, fit_predict, blend, predict_active
from src.submission import check
recipe = json.loads((RUN / 'recipe.json').read_text())
report = json.loads((RUN / 'report.json').read_text())
X, y = load()  # Rebuilds ignored feature caches from organizer files if necessary.
print('Labelled alerts / available features:', X.shape)
print('Positive rate:', y.mean())

## Features and frozen recipe
Amount distributions are calculated separately for each transaction direction, type, and direction × type pair. Relative features subtract the alert's overall mean and divide by its standard deviation; contrasts compare a pair with its parent type/direction. This removes differences in overall amount level and scale without using labels.

No target encoding, signal ID predictor, external labels, or post-alert transactions are used. Median imputation and scaling are fitted on each training fold. Missing categories keep a fixed schema; alerts without transactions remain present.

In [ ]:
pd.DataFrame([{'model': m['id'], 'estimator': m['model'], 'features': len(m['columns']),
               'weight': recipe['weights'][m['id']]} for m in recipe['models']])

In [ ]:
pd.DataFrame({
    'baseline': [report['baseline_development']['mean'], report['baseline_audit_auc']],
    'optimized': [report['candidate_development']['mean'], report['candidate_audit_auc']],
}, index=['Development CV (selection-biased)', 'Frozen audit split'])

## Reproduce the audit comparison
The recipe is already frozen. The cell below fits on development alerts and scores the same audit IDs without selecting any new features, parameters or weights. Tree models average the recorded five seeds; deterministic linear models are fitted once. Rank blends are ranking scores in [0, 1], not calibrated probabilities.

In [ ]:
dev, audit = split_indices(y)
legacy = json.loads((RUN / 'baseline' / 'summary.json').read_text())
old_predictions = {}
old_weights = {k: v for k, v in legacy['weights'].items() if v}
for name in old_weights:
    columns = legacy['selected_columns']
    spec = {'model': name, 'params': legacy['tuned_params'][name]}
    old_predictions[name] = fit_predict(spec, X.iloc[dev][columns], y.iloc[dev],
                                        X.iloc[audit][columns], recipe['fit_seeds'])
new_predictions = {}
for spec in recipe['models']:
    columns = spec['columns']
    new_predictions[spec['id']] = fit_predict(spec, X.iloc[dev][columns], y.iloc[dev],
                                             X.iloc[audit][columns], recipe['fit_seeds'])
old_score = roc_auc_score(y.iloc[audit], blend(old_predictions, old_weights, 'rank'))
new_score = roc_auc_score(y.iloc[audit], blend(new_predictions, recipe['weights'], recipe['method']))
assert abs(old_score - report['baseline_audit_auc']) < 1e-8
assert abs(new_score - report['candidate_audit_auc']) < 1e-8
print(f'Baseline audit AUC: {old_score:.6f}')
print(f'Optimized audit AUC: {new_score:.6f}')
print(f'Paired gain: {new_score - old_score:+.6f}')
print('Paired bootstrap 95% interval:', report['audit_gain_bootstrap_95pct'])

## Reproduce the submission
Refit the selected recipe on all 14,000 labelled alerts and predict every test alert. The previous CSV is preserved in the baseline archive. The test labels are unavailable, so leaderboard performance is not measured here.

In [ ]:
predict_active()
submission = pd.read_csv(config.OUTPUTS / config.SUBMISSION_NAME)
expected = pd.read_csv(config.TEST_SIGNALS)[config.ID]
checks = check(submission, expected)
assert all(passed for _, passed, _ in checks)
print('Submission checks:', checks)
submission.head()